# Unpack — Domain-Specific Knowledge Graph (DSKG)
### Complete Python Implementation for Google Colab

This notebook provides a complete implementation of the **Unpack DSKG**, replicating the Go-based neuro-symbolic anchor. It loads a two-layer Knowledge Graph (T-Box and A-Box) onto Neo4j.

**ACADEMIC FRAMING (Hogan et al. 2021):**
*   **T-Box (Terminology Box):** Formal ontology (Class, Relation, Property, Curriculum).
*   **A-Box (Assertion Box):** Instance data (Concepts, Formulas, L2Labels, Videos, UseCases).


### 1. Install Dependencies and Import Libraries
Install the `neo4j` python driver and `pandas` for data handling.


In [ ]:
!pip install neo4j pandas --quiet

import os
import time
import pandas as pd
from datetime import datetime
from dataclasses import dataclass
from typing import List, Optional, Dict, Any
from neo4j import GraphDatabase, Driver

# Configuration for Colab
DATA_DIR = "./data/raw" # Update this to your Google Drive path if needed
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "yourpassword"


### 2. Configure Neo4j Connection
Configure the Neo4j driver using credentials.


In [ ]:
def get_driver(uri, user, pwd):
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    try:
        driver.verify_connectivity()
        print(f"✓ Connected to Neo4j at {uri}")
        return driver
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        return None

# Use environment variables if available
uri = os.getenv("NEO4J_URI", NEO4J_URI)
user = os.getenv("NEO4J_USER", NEO4J_USER)
pwd = os.getenv("NEO4J_PASSWORD", NEO4J_PASSWORD)

driver = get_driver(uri, user, pwd)

def clear_all(tx):
    tx.run("MATCH ()-[r]-() DELETE r")
    tx.run("MATCH (n) DELETE n")
    print("🧹 Cleared all data")


### 3. Initialize Database Schema (Constraints and Indexes)
Create uniqueness constraints and full-text indexes for efficient retrieval.


In [ ]:
def create_schema(tx):
    statements = [
        "CREATE CONSTRAINT concept_id_unique IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT formula_id_unique IF NOT EXISTS FOR (f:Formula) REQUIRE f.id IS UNIQUE",
        "CREATE CONSTRAINT l2label_id_unique IF NOT EXISTS FOR (l:L2Label) REQUIRE l.id IS UNIQUE",
        "CREATE CONSTRAINT video_id_unique IF NOT EXISTS FOR (v:VideoResource) REQUIRE v.id IS UNIQUE",
        "CREATE CONSTRAINT usecase_id_unique IF NOT EXISTS FOR (u:UseCase) REQUIRE u.id IS UNIQUE",
        "CREATE CONSTRAINT class_name_unique IF NOT EXISTS FOR (c:Class) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT relation_name_uniq IF NOT EXISTS FOR (r:Relation) REQUIRE r.name IS UNIQUE",
        "CREATE CONSTRAINT property_name_uniq IF NOT EXISTS FOR (p:Property) REQUIRE p.name IS UNIQUE",
        "CREATE CONSTRAINT curriculum_id_uniq IF NOT EXISTS FOR (c:Curriculum) REQUIRE c.id IS UNIQUE",
        "CREATE INDEX concept_type IF NOT EXISTS FOR (c:Concept) ON (c.type)",
        "CREATE INDEX concept_layer IF NOT EXISTS FOR (c:Concept) ON (c.curriculum_layer)",
        "CREATE INDEX l2label_lang IF NOT EXISTS FOR (l:L2Label) ON (l.language_code)",
        "CREATE INDEX video_lang IF NOT EXISTS FOR (v:VideoResource) ON (v.language)",
        "CREATE INDEX video_diff IF NOT EXISTS FOR (v:VideoResource) ON (v.difficulty)",
        "CREATE INDEX formula_primary IF NOT EXISTS FOR (f:Formula) ON (f.is_primary)",
        "CREATE INDEX formula_level IF NOT EXISTS FOR (f:Formula) ON (f.display_level)",
        "CREATE INDEX usecase_domain IF NOT EXISTS FOR (u:UseCase) ON (u.domain)",
        "CREATE FULLTEXT INDEX concept_search IF NOT EXISTS FOR (c:Concept) ON EACH [c.name, c.description, c.core_theory]",
        "CREATE FULLTEXT INDEX l2label_search IF NOT EXISTS FOR (l:L2Label) ON EACH [l.label, l.description_l2, l.curriculum_name]",
        "CREATE FULLTEXT INDEX usecase_search IF NOT EXISTS FOR (u:UseCase) ON EACH [u.description, u.problem_example, u.domain]"
    ]
    for stmt in statements:
        tx.run(stmt)
    print("✓ Schema initialized: 9 constraints, 8 indexes, 3 fulltext indexes")


### 4. Load T-Box (Ontology Layer)
Load the formal ontology layer, defining Classes, Relations, and Properties.


In [ ]:
def load_ontology(tx):
    classes = [
        {"name": "Concept", "description": "A mathematical concept at undergraduate level.", "scope": "undergraduate mathematics", "valid_types": ["Foundation", "Calculus", "Geometry", "Rate of Change", "Goal", "Equation Type"]},
        {"name": "Formula", "description": "A formula node. Separate so UI agent fetches only formula on hover without loading full theory.", "scope": "mathematical notation"},
        {"name": "L2Label", "description": "A canonical multilingual label. Separate so multilingual agent queries by language_code without scanning all concept properties.", "scope": "multilingual localisation"},
        {"name": "VideoResource", "description": "An educational YouTube video. Separate so Agent 5 filters by language and difficulty independently of concept data.", "scope": "educational media"},
        {"name": "UseCase", "description": "A real-world application. Answers: why am I learning this? Shown in hover state 3.", "scope": "contextual scaffolding"},
        {"name": "ProblemType", "description": "A category of mathematics word problem (e.g. Related Rates). Links to Concept via EXEMPLIFIES and INVOLVES.", "scope": "problem classification"},
        {"name": "Curriculum", "description": "Academic curriculum providing scope context for the graph.", "scope": "academic scope"},
    ]
    for c in classes:
        tx.run("MERGE (n:Class {name: $p.name}) SET n = $p", p=c)

    relations = [
        {"name": "REQUIRES", "domain": "Concept", "range": "Concept", "definition": "Student cannot engage with target without mastering source. Asymmetric, transitive, weighted.", "is_transitive": True, "is_symmetric": False, "weight_meaning": "1.0=hard prerequisite, 0.8=recommended, 0.7=helpful"},
        {"name": "RELATED_TO", "domain": "Concept", "range": "Concept", "definition": "Two concepts co-occur in same problem or share mathematical structure. Neither is prerequisite.", "is_transitive": False, "is_symmetric": True, "weight_meaning": "1.0=definitionally linked, 0.8=frequently combined"},
        {"name": "HAS_FORMULA", "domain": "Concept", "range": "Formula", "definition": "Concept has a formula. Primary formula shown at hover state 1 (low CL).", "is_transitive": False, "is_symmetric": False},
        {"name": "HAS_LABEL", "domain": "Concept", "range": "L2Label", "definition": "Concept has canonical name in a specific language. Multilingual Agent B1 queries by language_code for textbook-canonical term.", "is_transitive": False, "is_symmetric": False},
        {"name": "HAS_RESOURCE", "domain": "Concept", "range": "VideoResource", "definition": "Concept has an educational video. Agent 5 attaches videos. UI queries by language for L1 content.", "is_transitive": False, "is_symmetric": False},
        {"name": "APPLIED_IN", "domain": "Concept", "range": "UseCase", "definition": "Concept applied in a real-world use case. Shown in hover state 3 (high support).", "is_transitive": False, "is_symmetric": False},
        {"name": "EXEMPLIFIES", "domain": "ProblemType", "range": "Concept", "definition": "Word problem of this type tests this Concept as the primary operation.", "is_transitive": False, "is_symmetric": False},
        {"name": "INVOLVES", "domain": "ProblemType", "range": "Concept", "definition": "Word problem of this type uses this Concept as a supporting operation.", "is_transitive": False, "is_symmetric": False},
        {"name": "COVERS", "domain": "Curriculum", "range": "Concept", "definition": "This Curriculum includes this Concept. Enables scope-bounded KG queries.", "is_transitive": False, "is_symmetric": False},
    ]
    for r in relations:
        tx.run("MERGE (n:Relation {name: $p.name}) SET n = $p", p=r)

    properties = [
        {"name": "id", "applies_to": "Concept", "datatype": "String", "definition": "Unique stable slug. Used as graph_node_id in Unpack UI JSON."},
        {"name": "core_theory", "applies_to": "Concept", "datatype": "String", "definition": "Strict mathematical definition shown at hover state 2 (medium CL)."},
        {"name": "type", "applies_to": "Concept", "datatype": "String", "definition": "Pedagogical category controlling UI token colour.", "valid_values": ["Foundation", "Calculus", "Geometry", "Rate of Change", "Goal", "Equation Type"]},
        {"name": "color", "applies_to": "Concept", "datatype": "String", "definition": "Hex colour for UI token underline and tooltip badge."},
        {"name": "curriculum_layer", "applies_to": "Concept", "datatype": "Integer", "definition": "0=pre-calc, 1=differentiation, 2=integration, 3=series, 4=multivariable."},
        {"name": "latex", "applies_to": "Formula", "datatype": "String", "definition": "Raw LaTeX rendered by KaTeX or MathJax in the progressive disclosure UI."},
        {"name": "notation_plain", "applies_to": "Formula", "datatype": "String", "definition": "ASCII plain-text formula shown when LaTeX rendering unavailable."},
        {"name": "display_level", "applies_to": "Formula", "datatype": "Integer", "definition": "1=hover (low CL), 2=click (medium CL)."},
        {"name": "language_code", "applies_to": "L2Label", "datatype": "String", "definition": "ISO 639-1 code. Multilingual Agent B1 queries on this field."},
        {"name": "text_direction", "applies_to": "L2Label", "datatype": "String", "definition": "ltr or rtl. UI sets CSS direction from this. Critical for Arabic."},
        {"name": "curriculum_name", "applies_to": "L2Label", "datatype": "String", "definition": "Textbook-canonical term in student language. Not a translation — a proper curriculum name."},
        {"name": "url", "applies_to": "VideoResource", "datatype": "String", "definition": "Direct YouTube video URL."},
        {"name": "rank_score", "applies_to": "VideoResource", "datatype": "Float", "definition": "Agent 5 semantic rank: channel authority × transcript similarity × duration filter."},
    ]
    for p in properties:
        tx.run("MERGE (n:Property {name: $p.name}) SET n = $p", p=p)

    curriculum = {
        "id": "undergrad_calculus",
        "name": "Undergraduate Calculus",
        "scope": "First and second year university mathematics",
        "covers_layers": [0, 1, 2, 3, 4],
        "study_context": "Unpack RQ3 — non-native English speaking STEM students",
        "citation": "Hogan et al. (2021). Knowledge Graphs. ACM Computing Surveys 54(4)."
    }
    tx.run("MERGE (cur:Curriculum {id: $id}) SET cur += $p", id=curriculum["id"], p=curriculum)
    print("✓ Ontology (T-Box) loaded: 7 Class, 9 Relation, 13 Property, 1 Curriculum")


### 5. Load A-Box (Instance Data: Concepts, Formulas, L2Labels, Videos, UseCases)
Batch upload instance data from CSV files and establish relationships.


In [ ]:
def load_concepts(tx, concepts_df):
    for _, row in concepts_df.iterrows():
        tx.run("""
            MERGE (c:Concept {id:$id})
            SET c.name=$name, c.description=$desc, c.core_theory=$theory,
                c.type=$type, c.color=$color, c.curriculum_layer=$layer,
                c.updated_at=datetime()
            """, id=row['concept_id'], name=row['concept_name'], desc=row['description'],
               theory=row['core_theory'], type=row['type'], color=row['color'], layer=int(row['curriculum_layer']))
    print(f"   ✓ {len(concepts_df)} Concept nodes")

def load_formulas(tx, formulas_df):
    for _, row in formulas_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (f:Formula {id:$id})
            SET f.name=$name, f.latex=$latex, f.notation_plain=$plain,
                f.display_level=$level, f.is_primary=$primary
            MERGE (c)-[:HAS_FORMULA {primary:$primary}]->(f)
            """, cid=row['concept_id'], id=row['formula_id'], name=row['formula_name'],
               latex=row['latex'], plain=row['notation_plain'],
               level=int(row['display_level']), primary=str(row['is_primary']).lower() == 'true')
    print(f"   ✓ {len(formulas_df)} Formula nodes + HAS_FORMULA edges")

def load_l2_labels(tx, labels_df):
    for _, row in labels_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (l:L2Label {id:$id})
            SET l.language_code=$lang, l.label=$label, l.description_l2=$desc,
                l.curriculum_name=$curriculum, l.text_direction=$dir
            MERGE (c)-[:HAS_LABEL]->(l)
            """, cid=row['concept_id'], id=row['label_id'], lang=row['language_code'],
               label=row['label'], desc=row['description_l2'],
               curriculum=row['curriculum_name'], dir=row['text_direction'])
    print(f"   ✓ {len(labels_df)} L2Label nodes + HAS_LABEL edges")

def load_videos(tx, videos_df):
    for _, row in videos_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (v:VideoResource {id:$id})
            SET v.platform=$platform, v.title=$title, v.url=$url,
                v.language=$lang, v.duration_sec=$dur, v.difficulty=$diff, v.channel=$ch
            MERGE (c)-[:HAS_RESOURCE]->(v)
            """, cid=row['concept_id'], id=row['video_id'], platform=row['platform'],
               title=row['title'], url=row['url'], lang=row['language'],
               dur=int(row['duration_sec']), diff=row['difficulty'], ch=row['channel'])
    print(f"   ✓ {len(videos_df)} VideoResource nodes + HAS_RESOURCE edges")

def load_use_cases(tx, use_cases_df):
    for _, row in use_cases_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (u:UseCase {id:$id})
            SET u.domain=$domain, u.description=$desc, u.problem_example=$example
            MERGE (c)-[:APPLIED_IN]->(u)
            """, cid=row['concept_id'], id=row['usecase_id'], domain=row['domain'],
               desc=row['description'], example=row['problem_example'])
    print(f"   ✓ {len(use_cases_df)} UseCase nodes + APPLIED_IN edges")

def load_edges(tx, edges_df):
    req, rel = 0, 0
    for _, row in edges_df.iterrows():
        etype = row['relationship_type']
        q = f"""
            MATCH (s:Concept {{id:$sid}}), (t:Concept {{id:$tid}})
            MERGE (s)-[r:{etype}]->(t)
            SET r.weight=$w, r.updated_at=datetime()
            """
        tx.run(q, sid=row['source_id'], tid=row['target_id'], w=float(row['weight']))
        if etype == 'REQUIRES': req += 1
        elif etype == 'RELATED_TO': rel += 1
    print(f"   ✓ {req} REQUIRES + {rel} RELATED_TO edges")

def link_curriculum(tx):
    res = tx.run("""
        MATCH (cur:Curriculum {id:"undergrad_calculus"}), (c:Concept)
        MERGE (cur)-[:COVERS {layer: c.curriculum_layer}]->(c)
        RETURN count(*) as n
        """)
    n = res.single()['n']
    print(f"   ✓ {n} COVERS edges linked to curriculum")


### 6. Verification Suite (Agent Access Pattern Tests)
Run 11 verification tests to ensure the Knowledge Graph is correctly loaded and supports various agent access patterns.


In [ ]:
def run_verification(tx):
    print("\n" + "="*62)
    print("  Verification — 11 tests across all agent access patterns")
    print("="*62)

    tests = [
        ("T01 T-Box: Class nodes ≥ 7", "MATCH (c:Class) RETURN count(c) as n", lambda r: r[0]['n'] >= 7),
        ("T02 T-Box: Relation nodes ≥ 9", "MATCH (r:Relation) RETURN count(r) as n", lambda r: r[0]['n'] >= 9),
        ("T03 T-Box: Property nodes ≥ 13", "MATCH (p:Property) RETURN count(p) as n", lambda r: r[0]['n'] >= 13),
        ("T04 A-Box: 39 Concept nodes", "MATCH (c:Concept) RETURN count(c) as n", lambda r: r[0]['n'] >= 39),
        ("T05 Prereq chain for related_rates", 'MATCH (:Concept{id:"related_rates"})-[:REQUIRES*1..6]->(p:Concept) RETURN count(distinct p) as n', lambda r: r[0]['n'] >= 7),
        ("T06 UI Agent: chain_rule primary formula", 'MATCH (:Concept{id:"chain_rule"})-[:HAS_FORMULA]->(f:Formula) WHERE f.is_primary=true AND f.display_level=1 RETURN f.notation_plain as f', lambda r: len(r) >= 1),
        ("T07 Multilingual: Sinhala for chain_rule", 'MATCH (:Concept{id:"chain_rule"})-[:HAS_LABEL]->(l:L2Label{language_code:"si"}) RETURN l.label as l', lambda r: len(r) >= 1),
        ("T08 Multilingual: Arabic direction is rtl", 'MATCH (:Concept{id:"related_rates"})-[:HAS_LABEL]->(l:L2Label{language_code:"ar"}) RETURN l.text_direction as d', lambda r: r[0]['d'] == "rtl"),
        ("T09 Agent 5: EN video for related_rates", 'MATCH (:Concept{id:"related_rates"})-[:HAS_RESOURCE]->(v:VideoResource{language:"en"}) RETURN v.title as t', lambda r: len(r) >= 1),
        ("T10 Context Agent: use case for optimization", 'MATCH (:Concept{id:"optimization"})-[:APPLIED_IN]->(u:UseCase) RETURN u.domain as d', lambda r: len(r) >= 1),
        ("T11 Curriculum COVERS all 39 Concepts", 'MATCH (:Curriculum{id:"undergrad_calculus"})-[:COVERS]->(c:Concept) RETURN count(c) as n', lambda r: r[0]['n'] >= 39),
    ]

    pass_count, fail_count = 0, 0
    for label, query, check in tests:
        try:
            res = list(tx.run(query))
            if check(res):
                print(f"  ✓ {label:48} {len(res)} row(s)")
                pass_count += 1
            else:
                print(f"  ✗ {label:48} FAILED")
                fail_count += 1
        except Exception as e:
            print(f"  ✗ {label:48} ERROR: {e}")
            fail_count += 1

    print("\n" + f"  Result: {pass_count} passed / {pass_count+fail_count} total")
    print("="*62)


### 7. Execution: Load Data and Run Tests
Run the loading functions in order.


In [ ]:
if driver:
    # 1. Read CSV Data
    try:
        concepts_df = pd.read_csv(f"{DATA_DIR}/concepts.csv")
        formulas_df = pd.read_csv(f"{DATA_DIR}/formulas.csv")
        labels_df = pd.read_csv(f"{DATA_DIR}/l2_labels.csv")
        videos_df = pd.read_csv(f"{DATA_DIR}/videos.csv")
        use_cases_df = pd.read_csv(f"{DATA_DIR}/use_cases.csv")
        edges_df = pd.read_csv(f"{DATA_DIR}/edges.csv")
        print("✓ CSV files read from " + DATA_DIR)
    except FileNotFoundError as e:
        print(f"✗ Files not found. Ensure CSVs are in {DATA_DIR}: {e}")
        # Stop execution
        raise e

    # 2. Run Database Operations
    with driver.session() as session:
        # Clear database
        session.execute_write(clear_all)

        start_time = time.time()

        # Step 1: Schema
        print("\nStep 1/9  Schema — constraints + indexes")
        session.execute_write(create_schema)

        # Step 2: Ontology (T-Box)
        print("Step 2/9  Ontology — T-Box (Class, Relation, Property, Curriculum)")
        session.execute_write(load_ontology)

        # A-Box Loading
        print("Step 3/9  Concepts — 39 :Concept nodes")
        session.execute_write(load_concepts, concepts_df)

        print("Step 4/9  Formulas — :Formula + HAS_FORMULA")
        session.execute_write(load_formulas, formulas_df)

        print("Step 5/9  L2 Labels — :L2Label + HAS_LABEL")
        session.execute_write(load_l2_labels, labels_df)

        print("Step 6/9  Videos — :VideoResource + HAS_RESOURCE")
        session.execute_write(load_videos, videos_df)

        print("Step 7/9  Use Cases — :UseCase + APPLIED_IN")
        session.execute_write(load_use_cases, use_cases_df)

        print("Step 8/9  Edges — REQUIRES + RELATED_TO")
        session.execute_write(load_edges, edges_df)

        print("Step 9/9  Curriculum — COVERS edges to all :Concept")
        session.execute_write(link_curriculum)

        elapsed = time.time() - start_time
        print("\n" + "="*62)
        print("  Unpack DSKG — Build Complete")
        print(f"  Build time: {elapsed:.2f} seconds")
        print("="*62)

        # Final Verification
        session.execute_read(run_verification)

    driver.close()
else:
    print("Driver not initialized. Please verify your credentials and connection.")
